In [ ]:
import numpy as np
import shap
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

In [ ]:
def loadDataSet(fileName):
    xArr = []  
    yArr = []  
    try:
        fr = pd.read_csv(fileName)
    except:
        fr = pd.read_excel(fileName)

    fr = fr.dropna()

    for i in range(len(fr)):
        line = fr.iloc[i].values  
        lineArr = [float(line[i]) for i in range(1, len(line)-1)]  
        target = float(line[-1])  
        
        xArr.append(lineArr)
        yArr.append(target)
    return np.array(xArr), np.array(yArr)


In [ ]:
random_state  = 27

# 加载数据集
xArr, yArr = loadDataSet('MY.xlsx')

# 首先将数据划分为训练集和临时集（后续用于验证和测试）
data_train_x, data_test_x, data_train_y, data_test_y = train_test_split(xArr, yArr, test_size=0.2, random_state=random_state)


In [ ]:
# 创建和训练 XGBoost 回归模型
xgb_regressor = xgb.XGBRegressor(n_estimators=200,max_depth=10,learning_rate=0.2,reg_lambda=5,n_jobs=-1)
xgb_regressor.fit(data_train_x, data_train_y)


In [ ]:
feature_label=["b", "d", "c", "Ls", "Ls/d", "n", "dl", "ds", "pl", "ps", "fy", "fs", "fc", "s", "DI"]
# feature_label=["b", "d", "c", "$L/s$", "$L/s$/d", "n", "$d/l$", "$d/s$", "$ρ/l$", "$ρ/s$", "$f/y$", "$f/s$", "$f/c$", "s", "DI"]

data_train_x_lable = pd.DataFrame(data_train_x,columns=feature_label)
data_train_x_lable

In [ ]:
# 创建SHAP解释器
explainer = shap.TreeExplainer(xgb_regressor, approximate=True) 

# 计算SHAP值
shap_values = explainer.shap_values(data_train_x_lable)


In [ ]:
shap.summary_plot(shap_values, data_train_x_lable,feature_names=feature_label,show = False)

plt.rcParams['font.family'] = 'Times New Roman'
plt.xlabel('', fontsize=22) 
plt.xticks(fontsize=18)  
# plt.gca().get_yaxis().set_visible(False)  

ax = plt.gca()

for spine in ax.spines.values():
    spine.set_visible(True)      
    spine.set_color('black')     
    spine.set_linewidth(1.5)       

fig = plt.gcf()  
cbar = fig.axes[-1]  

# 设置色条字体大小
cbar.tick_params(labelsize=18)  
cbar.set_ylabel('', fontsize=10)  

# plt.savefig("LAMA_shap_summary_plot.jpg", dpi=600, bbox_inches='tight')

plt.show